##### Importação dos Pacotes

In [0]:
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import monotonically_increasing_id, current_timestamp, date_format
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

##### Diretórios Catalog

In [0]:
path_bronze          = '/Volumes/01_bronze/schemas/tse/perfil_eleitorado'
path_silver          = '/Volumes/02_silver/schemas/tse/perfil_eleitorado'
path_gold_dimensao   = '/Volumes/03_gold/schemas/dimensao'

##### Consolida os arquivos da camada Silver (fato)

In [0]:
# Consolida arquivos na camada Silver (fato) em um único Dataframe
df_pel = spark.read.format("delta").load(path_silver)


In [0]:
# Validação dos esquemas das colunas no DataFrame
#df_pel.printSchema()

##### Salvar arquivo: tabela Temporária

In [0]:
# Salva o DataFrame em uma tabela Temporária (para leitura SQL)
df_pel.createOrReplaceTempView("df_pel")

In [0]:
%sql
-- Validação dos campos da dimensão
--SELECT DISTINCT
--    CD_ESTADO_CIVIL,
--    DS_ESTADO_CIVIL
--FROM df_pel

##### Criação da Dimensão: Estado Civil

In [0]:
# Criação de DataFrame da query para dimensão: Estado Civil
df_est_civil = spark.sql("""
                    SELECT DISTINCT
                    CASE
                      WHEN CD_ESTADO_CIVIL = -3 THEN 0
                      WHEN CD_ESTADO_CIVIL = 0 THEN 0
                      WHEN CD_ESTADO_CIVIL = 1 THEN 1
                      WHEN CD_ESTADO_CIVIL = 3 THEN 2
                      WHEN CD_ESTADO_CIVIL = 7 THEN 3
                      WHEN CD_ESTADO_CIVIL = 9 THEN 3
                      WHEN CD_ESTADO_CIVIL = 5 THEN 4
                    END AS SkEstadoCivil,
                    CASE
                      WHEN DS_ESTADO_CIVIL IN ('#NE') THEN 'Nao Informado'
                      WHEN DS_ESTADO_CIVIL IN ('NÃO INFORMADO') THEN 'Nao Informado'
                      WHEN DS_ESTADO_CIVIL IN ('SOLTEIRO') THEN 'Solteiro'
                      WHEN DS_ESTADO_CIVIL IN ('CASADO') THEN 'Casado'
                      WHEN DS_ESTADO_CIVIL IN ('SEPARADO JUDICIALMENTE') THEN 'Divorciado'
                      WHEN DS_ESTADO_CIVIL IN ('DIVORCIADO') THEN 'Divorciado'
                      WHEN DS_ESTADO_CIVIL IN ('VIÚVO') THEN 'Viuvo'
                    END AS DsEstadoCivil
                    FROM df_pel
                      """)

In [0]:
ordem_col = Window.orderBy("SkEstadoCivil")

In [0]:
ordem_col = Window.orderBy("SkEstadoCivil")

# Cria coluna Id (incremental)
df_est_civil = df_est_civil.withColumn("IdEstadoCivil",
                                        row_number().over(ordem_col)) \
                            .withColumn("DtAtualizacao",
                                       date_format(current_timestamp(), "dd/MM/yyyy HH:mm:ss"))

In [0]:
# Ordenando colunas na Dimensão
df_est_civil = df_est_civil.select("IdEstadoCivil", "SkEstadoCivil", "DsEstadoCivil", "DtAtualizacao")

In [0]:
# Validação do DataFrame
#display(df_est_civil)

##### Salvar DataFrame da dimensão: Faixa Etária na camada Gold

In [0]:
# Salva DataFrame (dimensão: Faixa Etária) na camada Gold
df_est_civil.write\
                .mode('overwrite')\
                .format('delta')\
                .option('mergeSchema', 'true')\
                .save(f'{path_gold_dimensao}/dim_estado_civil')

##### Fim